In [ ]:
# --! include root folder into PYTHONPATH --!

import os
import sys

thisdir = os.getcwd()
rootdir = os.path.abspath(os.path.join(thisdir, '..', '..'))
sys.path.append(rootdir)

# --! import Python libraries --!

import torch
import torch.nn.functional as F
import random
import numpy as np
import matplotlib.pyplot as plt

import util_data

In [ ]:
class expert(torch.nn.Module):
    def __init__(self, in_dim, out_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

class moe_dynamics(torch.nn.Module):
    def __init__(self, state_dim, action_dim, n_experts=4):
        super().__init__()

        self.in_dim = state_dim + action_dim
        self.out_dim = state_dim

        self.experts = nn.ModuleList([
            Expert(self.in_dim, self.out_dim)
            for _ in range(n_experts)
        ])

        # gating network
        self.gate = nn.Sequential(
            nn.Linear(self.in_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_experts)
        )

    def forward(self, s, a):
        x = torch.cat([s, a], dim=-1)

        # gating weights
        logits = self.gate(x)
        weights = F.softmax(logits, dim=-1)  # (B, K)

        # expert outputs
        outputs = torch.stack(
            [expert(x) for expert in self.experts],
            dim=-1
        )  # (B, state_dim, K)

        # weighted sum
        weights = weights.unsqueeze(1)  # (B, 1, K)
        out = (outputs * weights).sum(-1)

        return out
